# DAS protocol

* The network is formed of a single builder that is generating the block with all samples and `N` nodes distributed between validators and regular nodes.
* Builder knows all the validators
* All nodes have a unique 256 bits identifier, following the Kademlia procotol used in DEVP2P.
* We assume DAS protocol works on top of Kademlia DHT. We reuse identifiers and we can use the DHT to discover peers. 
However it is not necessary to use DEVP2P or Kademlia as long as we have nodes with random uniformly distributed identifiers and a way to discover nodes and resolve nodes identifiers to connection parameters.
* Builder divides the hash space into 512*512 sample-specific regions, according to the block matrix.
* Builder transforms the 2D block matrix into a two 1D line of samples (1 line row-wise, and 1 line column-wise, i.e, 1 line row-wise means that after all samples in the first row we continue by the first sample of the second row. 1 line column-wise we mean we first start with all the samples of the first column and then we continue with the samples of the second column)
* We assign each sample with a 256 identifier in the hash space with the double 1D aligment. Every sample is defined by a double id in the hash space, one following row-wise and another column-wise order. This way when fetching rows we can find colocated samples, but also when fetching columns.
* Builder chooses the redundancy factor `redundancy` - the higher `redundancy` the higher overhead but the more resistant the scheme becomes to malicious validators.
* The region of samples is defined as all the validators such that `dict(c, v) <= ((2^256 -1) * redundancy * 2) / N_validators `
* The validator pushes each sample to all the validators within the sample's region (Note that it's done in batches - i.e., the builder connects to each validator and gives them all the samples they should hold).

* When a new block is released, nodes got the samples from builder, according to the redundancy parameter and their identifier. 
After getting the assigned samples, they start the sampling process.
* There are two types of sampling processes: validator rows/columns fetching and random sampling.
* Validator row/column consists in getting 2 rows and 2 columns of the block matrix, selected randomly.
* Validators perform rows/columns fetching and regular nodes random sampling only.
* The rows/columns fetching and random sampling follows the same process with the only difference of the selection of the samples to obtain. 
We call sampling nodes, the nodes requesting samples, and serving nodes the nodes replying with samples:
    - Sampling nodes select a subset of replying nodes from a list of known nodes that are within the redundancy range following the id space for the samples requesting (how all nodes of the network are discovered is not included in this initial report, but will be in the following reports. Validators are supposed to know all other validators by default).
    - Sampling nodes send samples requests to the replying nodes selected. Samples requests include the wanted samples (all samples pending to be received are included in the requests).
    - Replying nodes reply with the samples requested they have, and also with known nodes to the same distance, from their nodes discovery table. This way nodes can easily discover kew nodes in the network.
    - Once a response is received from all replying nodes, or a configurable time out happened, the process is restarted selecting a new subset of replying nodes, in case there are still samples to be received. The number of replying nodes selected is increasing each round,  defined by an algorithm increasing an aggressiveness parameter each round.
    - Once all nodes known nodes are queried, extra nodes are added to the list and the process is repeated.
    - This extra nodes added consists of:
        - In case of row/column, validators in the range of the column id for each missing sample, in case of row fetching, or each row id in case of column sampling.
        - In case of random samplings, validators that requested in the range of the column or row id, even though not the specific sample.
    - For every increase of the aggressiveness params, validators increase the number of samples obtained from the builder, to protect from sybil attacks.



## Malicious protocol (ignoring requests / sybil attack)

* Some of both validator and regular nodes might be malicious. 
* Malicious behaviour is just ignoring samples requests without sending any response.


## Simulation parameters

* Number of nodes: 15000
* Number of attackers: Range from 0% to 90%
* Number of validators: 50% nodes are validators
* Redundancy parameter: 2
* Latency between nodes: 5ms - 125ms
* Number of samples per block : 512x512
* Sample size: 512 bytes
* Blocks: 1 blocks simulation.
* Number of random samples fetch: 75. 

The following cell is just loading traces files into a dataframe

In [2]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

ops_path = {'DAS': '../logsDasEvil0/operation.csv', 
            'Evil-0.25': '../logsDasEvil0.25/operation.csv',
            'Evil-0.5': '../logsDasEvil0.5/operation.csv',
            'Evil-0.75': '../logsDasEvil0.75/operation.csv',
            'Evil-0.9': '../logsDasEvil0.9/operation.csv'
           }
msgs_path = {'DAS': '../logsDasEvil0/messages.csv', 
            'Evil-0.25': '../logsDasEvil0.25/messages.csv',
            'Evil-0.5': '../logsDasEvil0.5/messages.csv',
            'Evil-0.75': '../logsDasEvil0.75/messages.csv',
            'Evil-0.9': '../logsDasEvil0.9/messages.csv'
          }


builder_address = '83814183170291850251680823880522715558189094423550585243365458794131648333116'

op_df={}
msg_df={}
for key in ops_path:
    op_df[key] = pd.read_csv(ops_path[key],index_col=False,low_memory=False)
for key in msgs_path:
    msg_df[key] = pd.read_csv(msgs_path[key],index_col=False,low_memory=False)


In [ ]:


fig3, ax3 = plt.subplots()

data = []
for key in op_df:

    vsdf = op_df[key].loc[(op_df[key]['type'] == 'ValidatorSamplingOperation')]
    data.append(vsdf['completion_time']/1000)

ax3.boxplot(data)

#ax3.legend()
#ax18.set_xlim([0,2])
ax3.set_ylim([0,12])
ax3.set_xticklabels([0,25,50,75,90])
ax3.set_title("Validator row/column fetching time (seconds)")
ax3.set_xlabel("% Malicious nodes")

In this graph we observe the time required to complete the row/column fetching process in validators. All validators are able to complete the row/colum fetching in our simulations. We observe the time increaseas, as the number of sybil in the network increases, but without preventing validators to complete the process. In all cases the process is completed within the 4 seconds requirements, but in the case of 90% sybils, where 75% of processes are completed within 5 seconds. 

In [ ]:

fig3, ax3 = plt.subplots()

data = []
for key in op_df:

    vsdf = op_df[key].loc[(op_df[key]['type'] == 'RandomSamplingOperation')]
    data.append(vsdf['completion_time']/1000)

ax3.boxplot(data)


ax3.set_ylim([0,15])
ax3.set_xticklabels([0,25,50,75,90])
ax3.set_title("Random sampling time regular nodes (seconds)")
ax3.set_xlabel("% Malicious nodes")

In this graph we observe the time required for the random sampling to complete when increasing the number of sybils. We can observe that only in case of 90% sybils, a small number of nodes are slighly delayed and cannot complete within the 12 seconds block time

In [ ]:

fig3, ax3 = plt.subplots()

data = []
for key in msg_df:

    vsdf = msg_df[key].loc[(msg_df[key]['nodeType'] == 'builder')]
    data.append(vsdf['bytesOut'].values[0]/1024/1024/1024)

ax3.bar([1,2,3,4,5],data)

ax3.set_xticklabels([0,0,25,50,75,90])
ax3.set_title("GBs sent by the builder per sampling process")
ax3.set_xlabel("% Malicious nodes")

In this graph we observe the amount of data sent by the builder per block in GBs. We observe this amount is reduced with the increasing number of sybils, since there are less validators requesting samples from the builder.

In [ ]:
fig3, ax3 = plt.subplots()

data = []
for key in msg_df:

    vsdf = msg_df[key].loc[(msg_df[key]['nodeType'] == 'validator')]
    data.append(vsdf['bytesOut']/1024/1024)

ax3.boxplot(data)


ax3.set_xticklabels([0,25,50,75,90])
ax3.set_title("MBs sent per validator")
ax3.set_xlabel("% Malicious nodes")

In this case we observe the amount of data sent by validators. We observe this is not incremented in our simulations, although the cause is that the number of validators requesting is also reduced for the presence of sybils. In any case, we observe the number of data sent is not reduced in the same range, meaning honest validators need to do more work in case of sybils.

In [ ]:
fig3, ax3 = plt.subplots()

data = []
for key in msg_df:

    vsdf = msg_df[key].loc[(msg_df[key]['nodeType'] == 'regular')]
    data.append(vsdf['bytesOut']/1024/1024)

ax3.boxplot(data)

ax3.set_xticklabels([0,25,50,75,90])
ax3.set_title("MBs sent per slot regular node")
ax3.set_xlabel("% Malicious nodes")

This is the data sent by regular nodes in MB. This amount of data is smaller compared to validators, since nodes validators tend to prioritize nodes withe more samples when fetching. Also, regular nodes have less capacity, and therefore samples sent by them may arrive later on, limiting their participation.


## Malicious builder (withhold attack)

* Builder withholds some specific samples from the matrix, without sending them to validators.


## Simulation parameters

* Number of nodes: 5000
* Number of validators: 50% nodes are validators
* Redundancy parameter: 2
* Latency between nodes: 5ms - 125ms
* Number of samples per block : 512x512
* Sample size: 512 bytes
* Blocks: 1 blocks simulation.
* Number of random samples fetch: 75. 
* 75% samples withheld by builder.

In [15]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

ops_path = {'DAS': '../logsDasEvil0.5000/operation.csv', 
            'Evil': '../logsDasBuilderEvil5000/operation.csv'
           }
msgs_path = {'DAS': '../logsDasEvil0.5000/messages.csv', 
            'Evil': '../logsDasBuilderEvil5000/messages.csv'
          }


builder_address = '83814183170291850251680823880522715558189094423550585243365458794131648333116'

op_df={}
msg_df={}
for key in ops_path:
    op_df[key] = pd.read_csv(ops_path[key],index_col=False,low_memory=False)
for key in msgs_path:
    msg_df[key] = pd.read_csv(msgs_path[key],index_col=False,low_memory=False)


In [ ]:


fig3, ax3 = plt.subplots()

data = []
for key in op_df:

    vsdf = op_df[key].loc[(op_df[key]['type'] == 'ValidatorSamplingOperation')]
    data.append(vsdf['completion_time']/1000)

ax3.boxplot(data)


ax3.set_ylim([0,12])
ax3.set_xticklabels([0,75])
ax3.set_title("Validator row/column fetching time (seconds)")
ax3.set_xlabel("% Withhold")

In [ ]:
fig3, ax3 = plt.subplots()

data = []
for key in op_df:

    vsdf = op_df[key].loc[(op_df[key]['type'] == 'RandomSamplingOperation')]
    #vsdf = op_df[key].loc[(op_df[key]['validator'] == 'no')]
    data.append(vsdf['completion_time']/1000)

ax3.boxplot(data)

#ax3.legend()
#ax18.set_xlim([0,2])
ax3.set_ylim([0,15])
ax3.set_xticklabels([0,75])
ax3.set_title("Random sampling time regular nodes (seconds)")
ax3.set_xlabel("% Withhold")

In [ ]:

fig3, ax3 = plt.subplots()

data = []
for key in msg_df:

    vsdf = msg_df[key].loc[(msg_df[key]['nodeType'] == 'builder')]
    data.append(vsdf['bytesOut'].values[0]/1024/1024/1024)

ax3.bar([1,2],data)

ax3.set_xticks([1,2])
ax3.set_xticklabels([0,75])
ax3.set_title("GBs sent by the builder per sampling process")
ax3.set_xlabel("% Withhold")

In [ ]:
fig3, ax3 = plt.subplots()

data = []
for key in msg_df:

    vsdf = msg_df[key].loc[(msg_df[key]['nodeType'] == 'validator')]
    data.append(vsdf['bytesOut']/1024/1024)

ax3.boxplot(data)

ax3.set_xticklabels([0,75])
ax3.set_title("MBs sent per validator")
ax3.set_xlabel("% Withhold")

In [ ]:
fig3, ax3 = plt.subplots()

data = []
for key in msg_df:

    vsdf = msg_df[key].loc[(msg_df[key]['nodeType'] == 'regular')]
    data.append(vsdf['bytesOut']/1024/1024)

ax3.boxplot(data)


ax3.set_xticklabels([0,75])
ax3.set_title("MBs sent per slot regular node")
ax3.set_xlabel("% Withhold")